# VLQ (IQM Quantum Hardware via LEXIS & HEAppE) - Bell State Demo

This notebook demonstrates submitting and executing a quantum circuit (**Bell state**) on the real **VLQ** (IQM) quantum computer using the **LEXIS** test environment and the **QHEAppE** scheduler on the **LUMI** supercomputer.

- **LUMI Project**: `project_465003105`
- **Repository Branch**: [`40-integration-of-qscheduler---qheappe-client-v100`](https://github.com/It4innovations/quantum-as-a-service/tree/40-integration-of-qscheduler---qheappe-client-v100)
- **LEXIS Project**: `vlq_qaas_project`
- **Resource**: `VLQ-CZ-dev_qshceduler-alpha`
- **Backend**: `VLQ` (Production)
- **LEXIS API URL**: `https://api.lexis.tech`

### 0. Dependencies Note
When running under the registered Tykky kernel **`LUMI-IQ (Python 3.12 - VLQ Quantum)`**, all required packages (`qaas`, `iqm-client`, `matplotlib`) are already pre-installed in the container image. You can skip directly to Step 1.


In [ ]:
# Pre-installed in Tykky kernel. Only run if in a blank non-container environment:
# %pip install --extra-index-url https://opencode.it4i.eu/api/v4/projects/107/packages/pypi/simple matplotlib "qaas[iqm_qiskit] @ git+https://github.com/It4innovations/quantum-as-a-service.git@40-integration-of-qscheduler---qheappe-client-v100"


### 1. Imports and Interactive Browser Login (OAuth Device Flow)
On LUMI Open OnDemand, you can authenticate **interactively in your browser**:
- If no token is provided, `QProvider` automatically initiates the **OAuth Device Flow**.
- It prints a URL and an authentication code in the cell output.
- You click the link, log in with **MyAccessID / LEXIS**, approve access, and the notebook continues automatically!

In [ ]:
import os
from qiskit import QuantumCircuit
from qaas.client import QProvider

# OAuth 2.0 Device Code Flow authentication
token = os.getenv("LEXIS_TOKEN")

if token:
    print("Using token from LEXIS_TOKEN environment variable.")
else:
    print("OAuth 2.0 Device Code Flow active.")
    print("-> QProvider will launch the interactive browser login in Step 3.")


### 2. Build the Quantum Circuit (Bell State)
Create a 2-qubit circuit with 3 classical measurement bits:
1. Apply Hadamard gate $H$ on qubit 0
2. Apply CNOT gate ($CX$) from qubit 0 to qubit 1 (creating entangled Bell state $\frac{|00\rangle + |11\rangle}{\sqrt{2}}$)
3. Measure qubits 0 and 1 into classical bits 0 and 1
4. Apply Hadamard gate $H$ on qubit 0 and measure into classical bit 2

In [ ]:
# 1. Build a Qiskit circuit (Bell state)
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

print(qc.draw("text"))


### 3. Initialize QProvider (Interactive Browser Login)
Connecting to the LEXIS testing API (`https://api.lexis.tech`) for project `vlq_qaas_project`.

> **Note**: Click the login link and enter the code shown in the cell output below.

In [ ]:
# 2. Instantiate QProvider with your LEXIS project
lexis_project = os.getenv("LEXIS_PROJECT", "vlq_qaas_project")
resource_name = os.getenv("LEXIS_RESOURCE", "VLQ-CZ-dev_qshceduler-alpha")

print(f"Connecting to LEXIS project: {lexis_project}...")
provider = QProvider(
    lexis_project=lexis_project,
    token=token if token else None,
)
print(f"QProvider successfully authenticated for project: {lexis_project}")


### 4. Initialize Production VLQ Backend and Transpile Circuit
Retrieve the production `VLQ` backend and transpile the circuit to native VLQ (IQM Star topology) basis gates and coupling map.

In [ ]:
# 3. Initialize the QBackend with the client
backend = provider.get_backend(
    lexis_resource=resource_name,
    backend_name="VLQ"  # Production VLQ backend!
)
print(f"Backend: {backend}")

# 4. Transpile the circuit for VLQ Star topology
qc_transpiled = backend.transpile(qc)
print("Circuit transpiled successfully for VLQ:")
print(qc_transpiled.draw("text"))

### 5. Submit Circuit Job
Submit the transpiled circuit to the QScheduler queue (1024 shots).

In [ ]:
# 5. Submit the circuit job
job = backend.run(qc_transpiled, shots=1024, job_name="demo_bell_state")
print(f"Job submitted successfully. Job ID: {job.job_id()}")

### 6. Await Job Execution and Retrieve Results
The call `job.result()` blocks until a terminal state (`FINISHED`, `FAILED`, `CANCELED`) is reached.
We extract results using both **legacy** Qiskit and **primitive** interfaces.

In [ ]:
# 6. Await job execution and retrieve results
print("Waiting for job execution to complete on VLQ...")
result = job.result()
counts = result.get_counts()

print("\n================ Execution Results ================")
print("Counts:", counts)


### 7. Visualize Measurement Results

In [ ]:
try:
    from qiskit.visualization import plot_histogram
    display(plot_histogram(counts, title="VLQ - Bell State Measurement Counts (1024 shots)"))
except Exception as err:
    print("Could not render histogram plot:", err)
    print("Counts:", counts)